# Submission 1 — Dataset, EDA and Related Work

**Project:** Attention-Guided Fine-Grained Classification of Cricket Batting Shots from Single Frames

**Task:** Given a single still frame of a batsman mid-stroke, predict which of four batting shots is being played — *drive*, *leg glance / flick*, *pull*, or *sweep*.

**Why single frames?** Almost all prior cricket shot work is video-based and leans on temporal models (LSTM/GRU over frame sequences). This project asks how much of the task survives when that temporal signal is removed, and whether spatial attention can recover part of the gap.

---

### What this notebook covers

| Section | Rubric item |
|---|---|
| A | Dataset information — name, source, licence, year, task type, counts, size, channels, format |
| B | Exploratory data analysis — class distribution, sample grid, image stats, duplicate audit, written findings |
| C | Related work summary — one row per paper |
| D | Submission checklist |

> **Before you push:** Kernel → Restart & Run All, so every plot and table is saved inside the `.ipynb`.

## Setup

In [ ]:
import os, sys, glob, json, hashlib, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 110,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "-",
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
import sklearn
print("sklearn:", sklearn.__version__)

In [ ]:
# ---- Locate the dataset -------------------------------------------------
# Kaggle  : add the dataset "aneesh10/cricket-shot-dataset" to the notebook,
#           it mounts at /kaggle/input/cricket-shot-dataset/data
# Colab   : upload + unzip, then point DATA_DIR at the extracted "data" folder

CANDIDATES = [
    "/kaggle/input/cricket-shot-dataset/data",
    "/kaggle/input/cricket-shot-dataset",
    "./data",
    "/content/data",
    "../input/cricket-shot-dataset/data",
]

DATA_DIR = None
for c in CANDIDATES:
    if os.path.isdir(c):
        subdirs = [d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))]
        if subdirs:
            DATA_DIR = c
            break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Add 'aneesh10/cricket-shot-dataset' via "
        "'+ Add Input' on Kaggle, or set DATA_DIR manually."
    )

CLASSES = sorted(d for d in os.listdir(DATA_DIR)
                 if os.path.isdir(os.path.join(DATA_DIR, d)))

print("DATA_DIR :", DATA_DIR)
print("classes  :", CLASSES)

## A. Dataset information

### A.1 Provenance

> Provenance fields below were taken from the dataset's Kaggle page (version 2). The numeric
> fields in A.3 are computed automatically — do not type those by hand.

| Field | Value |
|---|---|
| **Name** | Cricket Shots Dataset (Kaggle ref `aneesh10/cricket-shot-dataset`, version 2) |
| **Source** | `https://www.kaggle.com/datasets/aneesh10/cricket-shot-dataset` |
| **Author** | Aneesh Aparajit (Kaggle user `aneesh10`) |
| **Licence** | Unknown (as listed on the Kaggle page — no licence is granted) |
| **Year published** | 2022 (current version 2, updated 20 Sep 2022) |
| **Task type** | Multi-class, single-label image classification |
| **Number of classes** | 4 |
| **Class names** | `drive`, `legglance-flick`, `pullshot`, `sweep` |
| **Construction** | Still frames collected from internet sources, then expanded by offline augmentation |

### A.2 Why this dataset matters in its domain

Cricket is watched by well over a billion people, and broadcast and coaching workflows
around it are still heavily manual. Automatic batting-shot recognition feeds three
concrete applications that appear repeatedly in the literature: **highlight generation**
for broadcasters, who currently tag shot types by hand across many hours of footage;
**coaching and technique analysis**, where a shot-type label is the first step before any
biomechanical feedback can be given; and **automated commentary and context-aware
advertising** systems.

What makes it a *hard* classification problem rather than a trivial one is that all four
classes share the same scene grammar — one batsman, one pitch, the same kit, the same
stadium backdrops. The label is determined almost entirely by **body posture and bat
angle**, occupying a small fraction of the frame. Prior work makes exactly this point:
the similarity between batting shots is what makes hand-crafted feature extraction
impractical and motivates learned representations. That property is also why this
dataset suits the later submissions — it leaves genuine headroom for an attention
module to demonstrate a measurable effect, which a coarse or visually separable dataset
would not.

### A.3 Scan every image and build the inventory table

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

records = []
bad_files = []

for cls in CLASSES:
    cdir = os.path.join(DATA_DIR, cls)
    for fname in sorted(os.listdir(cdir)):
        ext = os.path.splitext(fname)[1].lower()
        if ext not in IMG_EXTS:
            continue
        fpath = os.path.join(cdir, fname)
        try:
            with Image.open(fpath) as im:
                w, h = im.size
                mode, fmt = im.mode, im.format
            records.append({
                "path": fpath,
                "file": fname,
                "label": cls,
                "width": w,
                "height": h,
                "aspect": round(w / h, 3),
                "mode": mode,
                "format": fmt,
                "kb": round(os.path.getsize(fpath) / 1024, 1),
            })
        except Exception as e:
            bad_files.append((fpath, repr(e)))

df = pd.DataFrame(records)

print(f"readable images : {len(df):,}")
print(f"corrupt/skipped : {len(bad_files)}")
if bad_files:
    for p, e in bad_files[:10]:
        print("   ", p, "->", e)

df.head()

In [ ]:
# ---- Headline dataset facts --------------------------------------------
n_total = len(df)
counts = df["label"].value_counts().reindex(CLASSES)
pct = (counts / n_total * 100).round(2)

summary = pd.DataFrame({
    "images": counts,
    "share_%": pct,
    "median_w": df.groupby("label")["width"].median().reindex(CLASSES).astype(int),
    "median_h": df.groupby("label")["height"].median().reindex(CLASSES).astype(int),
    "mean_kb": df.groupby("label")["kb"].mean().reindex(CLASSES).round(1),
})
summary.index.name = "class"

print("=" * 62)
print(f"{'TOTAL IMAGES':<22}: {n_total:,}")
print(f"{'CLASSES':<22}: {len(CLASSES)}")
print(f"{'FILE FORMATS':<22}: {dict(Counter(df['format']))}")
print(f"{'COLOUR MODES':<22}: {dict(Counter(df['mode']))}")
print(f"{'CHANNELS':<22}: {'3 (RGB)' if set(df['mode']) <= {'RGB'} else 'mixed - see modes above'}")
print(f"{'WIDTH  min/med/max':<22}: {df.width.min()} / {int(df.width.median())} / {df.width.max()}")
print(f"{'HEIGHT min/med/max':<22}: {df.height.min()} / {int(df.height.median())} / {df.height.max()}")
print(f"{'TOTAL SIZE':<22}: {df.kb.sum()/1024:.1f} MB")
print("=" * 62)

summary

## B. Exploratory data analysis

### B.1 Class distribution — is the data imbalanced?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

bar_colour = "#3d6b9c"
ax = axes[0]
bars = ax.bar(counts.index, counts.values, color=bar_colour, width=0.62)
ax.bar_label(bars, fmt="%d", padding=3, fontsize=9)
ax.set_title("Images per class", fontweight="bold", loc="left")
ax.set_ylabel("count")
ax.set_ylim(0, counts.max() * 1.16)
ax.tick_params(axis="x", rotation=15)
ax.axhline(counts.mean(), color="#c0392b", ls="--", lw=1.2,
           label=f"mean = {counts.mean():.0f}")
ax.legend(frameon=False, fontsize=9)

ax = axes[1]
ax.pie(counts.values, labels=counts.index, autopct="%1.1f%%",
       startangle=90, counterclock=False,
       colors=["#3d6b9c", "#5b8db8", "#7fb0d3", "#a8cde4"],
       wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax.set_title("Class share", fontweight="bold", loc="left")

plt.tight_layout()
plt.show()

imbalance_ratio = counts.max() / counts.min()
print(f"largest class  : {counts.idxmax()}  ({counts.max():,})")
print(f"smallest class : {counts.idxmin()}  ({counts.min():,})")
print(f"imbalance ratio: {imbalance_ratio:.2f} : 1")

if imbalance_ratio < 1.5:
    verdict = "BALANCED - plain accuracy is meaningful; no class weighting needed."
elif imbalance_ratio < 3:
    verdict = "MILDLY IMBALANCED - report macro-F1 alongside accuracy; consider class weights."
else:
    verdict = "STRONGLY IMBALANCED - use class weights or resampling; accuracy alone is misleading."
print("\nverdict        :", verdict)

### B.2 Sample image grid — 5 per class

The point of this grid is not decoration. Look for: how much of the frame the batsman
actually occupies, how much background clutter (crowd, sightscreen, advertising boards)
competes for attention, and whether any two classes look near-identical to your own eye.

In [ ]:
N_PER_CLASS = 5
rng = np.random.RandomState(RANDOM_SEED)

fig, axes = plt.subplots(len(CLASSES), N_PER_CLASS,
                         figsize=(2.3 * N_PER_CLASS, 2.5 * len(CLASSES)))

for r, cls in enumerate(CLASSES):
    paths = df.loc[df.label == cls, "path"].values
    picks = rng.choice(paths, size=min(N_PER_CLASS, len(paths)), replace=False)
    for c in range(N_PER_CLASS):
        ax = axes[r, c]
        ax.axis("off")
        if c < len(picks):
            with Image.open(picks[c]) as im:
                ax.imshow(im.convert("RGB"))
        if c == 0:
            ax.set_title(cls, loc="left", fontweight="bold",
                         fontsize=11, color="#3d6b9c", pad=6)

plt.suptitle("Random samples per class", fontweight="bold", y=1.002, x=0.01, ha="left")
plt.tight_layout()
plt.show()

### B.3 Image geometry

If the images are already a uniform size, resizing is trivial and you can note that.
If they vary, that variation decides your resize/crop strategy and is worth reporting.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

ax = axes[0]
ax.hist(df.width, bins=40, color="#3d6b9c", alpha=0.85)
ax.set_title("Width (px)", fontweight="bold", loc="left")
ax.set_xlabel("pixels"); ax.set_ylabel("images")

ax = axes[1]
ax.hist(df.height, bins=40, color="#5b8db8", alpha=0.85)
ax.set_title("Height (px)", fontweight="bold", loc="left")
ax.set_xlabel("pixels")

ax = axes[2]
ax.hist(df.aspect, bins=40, color="#7fb0d3", alpha=0.85)
ax.axvline(1.0, color="#c0392b", ls="--", lw=1.2, label="square")
ax.set_title("Aspect ratio (w/h)", fontweight="bold", loc="left")
ax.set_xlabel("ratio"); ax.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

n_unique_shapes = df.groupby(["width", "height"]).size().shape[0]
print(f"distinct (width, height) combinations: {n_unique_shapes}")
print("\ntop 10 most common sizes:")
print(df.groupby(["width", "height"]).size()
        .sort_values(ascending=False).head(10).to_string())

### B.4 Duplicate / near-duplicate audit

**This is the most important cell in the notebook.**

This dataset was built by collecting images and then *augmenting them offline*. That
means multiple files can be transformations of the same original photograph. If a random
train/test split puts one copy in train and another in test, the model is evaluated on
pictures it has effectively already seen, and every accuracy number from Submission 2
onward is inflated.

Below we compute a **difference hash (dHash)** for each image — a 64-bit perceptual
fingerprint that is robust to small changes in scale, compression and brightness but
sensitive to real differences in content. Images sharing a hash are near-duplicates.

We then group images into **duplicate groups** and recommend splitting by *group*
rather than by *file*, so no group is split across train and test.

In [ ]:
def dhash(path, hash_size=8):
    """64-bit perceptual difference hash. No external dependency."""
    with Image.open(path) as im:
        im = im.convert("L").resize((hash_size + 1, hash_size), Image.LANCZOS)
        a = np.asarray(im, dtype=np.int16)
    diff = a[:, 1:] > a[:, :-1]
    bits = diff.flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


print("hashing images ...")
df["dhash"] = [dhash(p) for p in df["path"]]
print("done.")

# --- exact perceptual collisions ---
hash_groups = df.groupby("dhash").indices
dup_groups = {h: idx for h, idx in hash_groups.items() if len(idx) > 1}

n_in_dupes = sum(len(v) for v in dup_groups.values())
n_redundant = n_in_dupes - len(dup_groups)

print(f"\nunique perceptual hashes : {df['dhash'].nunique():,} / {len(df):,} images")
print(f"duplicate groups         : {len(dup_groups):,}")
print(f"images inside a group    : {n_in_dupes:,}")
print(f"redundant copies         : {n_redundant:,}  ({n_redundant/len(df)*100:.1f}% of dataset)")

In [ ]:
# --- do duplicates cross class boundaries? (a labelling red flag) --------
cross_class = 0
for h, idx in dup_groups.items():
    if df.loc[idx, "label"].nunique() > 1:
        cross_class += 1

print(f"duplicate groups spanning MORE THAN ONE class: {cross_class}")
if cross_class:
    print("  -> these are label conflicts: identical images with different labels.")
    print("  -> inspect a few before deciding how to handle them.\n")
    shown = 0
    for h, idx in dup_groups.items():
        if df.loc[idx, "label"].nunique() > 1 and shown < 3:
            print(f"  group {h}:")
            print(df.loc[idx, ["file", "label"]].to_string(index=False))
            shown += 1
else:
    print("  -> good: every duplicate group sits inside a single class.")

In [ ]:
# --- assign a group id so the split can respect duplicate groups --------
group_of = {}
for gid, (h, idx) in enumerate(hash_groups.items()):
    for i in idx:
        group_of[i] = gid
df["group_id"] = pd.Series(group_of).reindex(df.index).values

print(f"split units (duplicate groups) : {df['group_id'].nunique():,}")
print(f"raw files                      : {len(df):,}")
print(f"\nEffective dataset size is {df['group_id'].nunique():,}, not {len(df):,}.")
print("Submissions 2-4 must split on group_id, never on row index.")

# visualise a couple of duplicate groups so the finding is evidenced
if dup_groups:
    biggest = sorted(dup_groups.items(), key=lambda kv: -len(kv[1]))[:3]
    ncol = max(len(idx) for _, idx in biggest)
    ncol = min(ncol, 6)
    fig, axes = plt.subplots(len(biggest), ncol,
                             figsize=(2.1 * ncol, 2.3 * len(biggest)))
    axes = np.atleast_2d(axes)
    for r, (h, idx) in enumerate(biggest):
        for c in range(ncol):
            ax = axes[r, c]; ax.axis("off")
            if c < len(idx):
                row = df.loc[idx[c]]
                with Image.open(row["path"]) as im:
                    ax.imshow(im.convert("RGB"))
                ax.set_title(row["label"], fontsize=8, color="#3d6b9c")
        axes[r, 0].set_ylabel(f"group {r+1}")
    plt.suptitle("Largest near-duplicate groups", fontweight="bold",
                 y=1.004, x=0.01, ha="left")
    plt.tight_layout(); plt.show()

### B.5 Proposed split for Submissions 2–4

Fix the split **here**, save it, and reuse the identical split in every later submission.
If the split changes between Submission 3 and Submission 4, the attention comparison is
worthless because the two models were not evaluated on the same data.

In [ ]:
# Split on duplicate GROUPS, stratified by the group's class.
grp = (df.groupby("group_id")
         .agg(label=("label", "first"), n=("label", "size"))
         .reset_index())

train_g, temp_g = train_test_split(
    grp, test_size=0.30, stratify=grp["label"], random_state=RANDOM_SEED)
val_g, test_g = train_test_split(
    temp_g, test_size=0.50, stratify=temp_g["label"], random_state=RANDOM_SEED)

split_of = {}
for name, part in [("train", train_g), ("val", val_g), ("test", test_g)]:
    for g in part["group_id"]:
        split_of[g] = name
df["split"] = df["group_id"].map(split_of)

pivot = (df.pivot_table(index="label", columns="split", values="path",
                        aggfunc="count")
           .reindex(CLASSES)[["train", "val", "test"]])
pivot["total"] = pivot.sum(axis=1)
print("Images per class per split (70 / 15 / 15, grouped + stratified):\n")
print(pivot.to_string())

# sanity check: no group may appear in two splits
leak = df.groupby("group_id")["split"].nunique().max()
print(f"\nmax splits touched by any one group: {leak}  (must be 1)")
assert leak == 1, "LEAKAGE: a duplicate group was split across partitions"

# Save inside the repo's artifacts/ folder when run from notebooks/,
# otherwise (e.g. on Kaggle) next to the notebook.
ART_DIR = "../artifacts" if os.path.isdir("../artifacts") else "."
MANIFEST = os.path.join(ART_DIR, "split_manifest.csv")
df[["path", "file", "label", "group_id", "split"]].to_csv(MANIFEST, index=False)
print(f"\nsaved -> {MANIFEST}   (load this in Submissions 2, 3 and 4)")

### B.6 Written findings

> **Replace every `___` with the number printed above before you submit.** Keep the
> prose; it is the part the rubric marks.

**1. Balance.** The dataset contains `___` images across four classes. The largest class
holds `___` images and the smallest `___`, an imbalance ratio of `___` : 1. This is
`[balanced / mildly imbalanced / strongly imbalanced]`. *Consequence:* accuracy alone
`[is / is not]` a sufficient headline metric, so macro-averaged F1 is reported alongside
it from Submission 2 onward, and class weights `[are / are not]` applied in Submission 3.

**2. Redundancy and leakage risk.** Perceptual hashing found `___` near-duplicate groups
covering `___` images, meaning `___` files (`___`%) are redundant copies of another
image. The effective dataset size is therefore `___` independent images rather than the
`___` files on disk. This is a direct consequence of the dataset being augmented offline
before publication. *Consequence:* all splits are made on duplicate-group identity, not
on file identity. A naive per-file random split would place augmented copies of the same
source photograph in both train and test, inflating reported accuracy. This is the single
largest threat to validity in this dataset and, as far as we can tell from published work
using it, is not usually controlled for.

**3. Class similarity.** All four classes share an identical scene grammar: one batsman
on a pitch, the same kit, similar stadium backdrops, frequently a crowd and sightscreen
behind. The discriminative signal is confined to body posture and bat angle, which
occupy a minority of the frame. *Consequence:* the hardest confusions are expected
between `___` and `___` (name the pair that looked most alike in the B.2 grid), and this
is the specific weakness the attention module in Submission 4 is intended to address —
by suppressing background response and concentrating capacity on the batsman.

**4. Image geometry and noise.** Images span `___`–`___` px in width and `___`–`___` px
in height, across `___` distinct size combinations, in `___` format with `___` channels.
Backgrounds vary in illumination and crowd density; some frames are low-resolution
broadcast captures with visible compression artefacts. *Consequence:* all images are
resized to 224×224 to match the ImageNet pretrained backbones used in Submission 2, with
normalisation to ImageNet channel statistics.

**5. Preprocessing and augmentation plan.** Based on the above:

| Decision | Choice | Driven by finding |
|---|---|---|
| Resize | 224 × 224 | #4 — size variation + pretrained backbone input |
| Normalisation | ImageNet mean/std | #4 — transfer learning in Submission 2 |
| Horizontal flip | **Not applied** | #3 — flipping converts an off-side shot into a leg-side one and corrupts the label |
| Rotation | ±10° | #4 — mild broadcast camera tilt is realistic |
| Brightness / contrast jitter | ±20% | #4 — varying stadium illumination |
| Random resized crop | scale 0.8–1.0 | #3 — forces reliance on the batsman, not framing |
| Class weights | `[yes / no]` | #1 — depends on the measured imbalance ratio |
| Split strategy | grouped + stratified, 70/15/15 | #2 — duplicate-group leakage |

> **Note the horizontal-flip decision.** It is the one augmentation most people apply
> reflexively, and it is wrong for this dataset: a sweep and a leg glance are defined
> partly by which side of the body the ball is played to. Mirroring the image changes the
> correct answer. Say this explicitly in your report — it shows the augmentation choices
> were reasoned about rather than copied from a tutorial.

## C. Related work summary

Papers only, no blog posts. Note that most prior work on cricket shot classification is
**video-based**; this is the gap the present project addresses by restricting itself to
single frames.

| # | Paper (author, year) | Dataset used | Model(s) | Approach | Result | Reference link |
|---|---|---|---|---|---|---|
| 1 | Sen, Deb, Dhar & Koshiba (2021), *Sensors* — "CricShotClassify" | CricShot10 (own; 10 shot classes, video) | Conventional CNN, dilated CNN, VGG16 / InceptionV3 / Xception / DenseNet169, each + GRU | Transfer learning with frozen backbones, then selective unfreezing of the final 4 and final 8 VGG16 layers; GRU for temporal dependency | 86% (all layers frozen, VGG16-GRU); 93% (final 4 or final 8 layers fine-tuned) | https://pmc.ncbi.nlm.nih.gov/articles/PMC8072636 |
| 2 | Kang (2025), *arXiv:2510.09187* — "Modern Deep Learning Approaches for Cricket Shot Classification" | CricShot10 | 7 models: CNN+GRU, Bi-LSTM + attention, ViT, transfer learning, EfficientNet-B0 + GRU | Unified re-implementation of prior work under one standardised evaluation protocol; proposes EfficientNet-B0 + GRU | 92.25% (proposed). **Re-implementations of prior published models scored 46.0%, 55.6% and 57.7% against originally reported 96%, 99.2% and 93%** | https://arxiv.org/abs/2510.09187 |
| 3 | Dey & Biswas (2024), *International Journal of Engineering* — "Shot-ViT" | CBSId (own; 2,160 annotated **images**, 7 classes) | Fine-tuned Vision Transformer, vs VGG19, ResNet50, I-AlexNet, ViT_B32 | Image-level (not video) classification; ViT fine-tuned for cricket batting shots | Shot-ViT outperforms all four CNN/ViT baselines compared | https://www.ije.ir/article_195819.html |
| 4 | Kumar et al. (2019), *IJRTE* — "Video based cricket shot classification using modified LRCN approach" | Own video dataset | Modified LRCN (CNN + LSTM) | CNN for spatial features, LSTM for temporal modelling | 96%, but on a reduced set of **5** shot classes rather than the full 10 | https://arxiv.org/abs/2510.09187 *(reported in the survey above; locate the original via your library)* |
| 5 | Bhat et al. (2024), *IJERCSE* — "Classification of cricket shots from cricket videos using deep learning models" | Cricket shot video dataset | CNN+RNN, attention network, ViT | Direct comparison of three deep learning paradigms on one dataset | 99.2% (CNN+RNN), 99.19% (attention), 98.9% (ViT) — figures not reproduced by independent re-implementation (see row 2) | https://arxiv.org/abs/2510.09187 *(reported in the survey above)* |
| 6 | Foysal, Islam, Karim & Neehal (2019), *arXiv:1909.01228* — "Shot-Net" / cricket bowler identification | BowlersNet (own; 8,100 **images**, 18 bowlers) | VGG16 + transfer learning | Single-frame image classification of cricket bowling actions; selective layer freezing | 93.3% average test accuracy | https://arxiv.org/abs/1909.01228 |

### What this table tells us

Three observations worth stating explicitly in the report:

1. **The field is overwhelmingly video-based.** Rows 1, 2, 4 and 5 all model temporal
   sequences. Only rows 3 and 6 operate on single images, and row 6 classifies bowlers
   rather than shots. Single-frame shot classification is therefore genuinely
   under-examined — which is the niche this project occupies.

2. **Reported accuracies in this area are unreliable.** Row 2 is a 2025 study that
   re-implemented prior models under a standardised protocol and recovered roughly half
   the originally published accuracy. The likely causes are inconsistent splits and
   evaluation protocols — precisely the failure mode that the duplicate-group audit in
   section B.4 is designed to avoid here. **This justifies the methodological care taken
   in this project and should be cited when presenting our own numbers.**

3. **Attention appears, but is never isolated.** Row 5 reports an attention network, but
   alongside architectural changes, so the contribution of attention itself cannot be
   separated. Submission 4 of this project holds everything else constant and varies only
   the attention module, which makes the measured difference attributable.

> **Before submitting:** open each reference, confirm the figures above against the
> original text, and replace the two placeholder links in rows 4 and 5 with the actual
> DOIs once you locate them through your institution's library.

## D. Submission checklist

- [ ] Dataset added to the notebook and `DATA_DIR` resolves without error
- [x] Licence and publication year filled in from the Kaggle page (section A.1)
- [ ] **Kernel → Restart & Run All**, so every plot and table is stored in the `.ipynb`
- [ ] All `___` placeholders in section B.6 replaced with the computed numbers
- [ ] The most-confused class pair named in finding #3, from looking at the B.2 grid
- [ ] `artifacts/split_manifest.csv` saved and committed — Submissions 2, 3 and 4 load this file, and do **not** re-split
- [ ] Reference links in the related-work table verified
- [ ] Notebook pushed to GitHub with outputs visible, and the link submitted

### Carrying forward

```python
# at the top of Submissions 2, 3 and 4:
manifest = pd.read_csv("../artifacts/split_manifest.csv")
train_df = manifest[manifest.split == "train"]
val_df   = manifest[manifest.split == "val"]
test_df  = manifest[manifest.split == "test"]
RANDOM_SEED = 42
```

Reusing this manifest unchanged is what makes the Submission 3 vs Submission 4 comparison
a controlled experiment rather than two unrelated training runs.